# 01 · Models, messages, and schemas — solutions

**Prerequisites:** Python functions, dictionaries, and exceptions.

**Learning objectives:** Construct typed messages; validate output shape; distinguish schema validity from evidence support.

**Guide companion:** sections 3–5 in `LANGCHAIN_LANGGRAPH_LEARNING_GUIDE.md` at the project root.

**How to work:** Run setup, implement each challenge, then run its acceptance cell. Starter functions deliberately raise `NotImplementedError`; this is expected until you complete them. Restart the kernel and run all cells after finishing. You do not need any other notebook or paid API calls. Budget about 45–90 minutes, or longer for the capstone.

This copy contains complete implementations and answer explanations. Try the exercise notebook first.


In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"

# Fictional test data, not real people, policies, or research sources.
NOTES = [
    {"id": "s1", "url": "fixture://architecture", "text": "Cedar uses LangGraph to route research tasks."},
    {"id": "s2", "url": "fixture://review", "text": "Cedar pauses its workflow for human review."},
    {"id": "s3", "url": "fixture://ownership", "text": "Mira maintains Cedar."},
    {"id": "s4", "url": "fixture://team", "text": "Mira works on team Atlas."},
    {"id": "s5", "url": "fixture://policy", "text": "Atlas reviews Cedar evidence monthly."},
]
BY_ID = {note["id"]: note for note in NOTES}

from langchain.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, ConfigDict, Field, ValidationError


## Challenge 1 · Construct a model input

Implement `prepare_messages(question, note)` returning exactly two LangChain message objects: a `SystemMessage` and a `HumanMessage`.
The system message must instruct the model to use only supplied evidence. The human message must contain the question, note ID, and note text. Do not invoke a model.
Example: passing the ownership note should make `s3` and `Mira maintains Cedar.` visible in the user message.


In [ ]:
def prepare_messages(question: str, note: dict) -> list:
    return [
        SystemMessage(content="Use only the supplied evidence. If it cannot answer the question, say so."),
        HumanMessage(content=f"Question: {question}\nNote {note['id']}: {note['text']}"),
    ]


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
messages = prepare_messages("Who maintains Cedar?", BY_ID["s3"])
assert len(messages) == 2, "Return exactly two messages"
assert isinstance(messages[0], SystemMessage), "The first message must be a system message"
assert isinstance(messages[1], HumanMessage), "The second message must be a human message"
assert all(s in messages[1].content for s in ["Who maintains Cedar?", "s3", BY_ID["s3"]["text"]]), "Include the question and identifiable evidence"
assert messages[0].content.strip(), "Provide an evidence-only instruction; inspect it manually"
print("PASS: message shape and evidence fields. Read the system instruction to check its meaning.")


### Why this works

Messages are data passed to a model; constructing them costs no API calls. A system instruction expresses desired behavior, but Python must enforce requirements that cannot depend on model compliance.


## Challenge 2 · Define a strict answer schema

Implement `make_claim_type()` returning a Pydantic `BaseModel` subclass with exactly two required fields: `text` and `source_id`.
Both must be strict, nonempty strings. Reject unknown fields. Then implement `parse_claim(payload)` returning an instance of that type; invalid payloads must raise `ValidationError`.
The factory keeps this exercise self-contained; defining the class at module level is also acceptable if your factory returns it.


In [ ]:
class Claim(BaseModel):
    model_config = ConfigDict(extra="forbid")
    text: str = Field(strict=True, min_length=1)
    source_id: str = Field(strict=True, min_length=1)

def make_claim_type() -> type[BaseModel]:
    return Claim

def parse_claim(payload: dict) -> BaseModel:
    return make_claim_type().model_validate(payload)


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
claim_type = make_claim_type()
assert issubclass(claim_type, BaseModel)
assert set(claim_type.model_fields) == {"text", "source_id"}
valid = parse_claim({"text": "Mira maintains Cedar.", "source_id": "s3"})
assert valid.source_id == "s3"
for payload in [
    {"text": "Mira maintains Cedar."},
    {"text": "", "source_id": "s3"},
    {"text": "x", "source_id": ""},
    {"text": "x", "source_id": 3},
    {"text": 42, "source_id": "s3"},
    {"text": "x", "source_id": "s3", "confidence": 1},
]:
    try:
        parse_claim(payload)
    except ValidationError:
        pass
    else:
        raise AssertionError(f"Schema should reject: {payload}")
print("PASS: strict schema and invalid-payload rejection")


### Why this works

A schema checks shape and types. Strict strings prevent unwanted coercion, and forbidden extra fields expose unexpected output. None of these checks establishes that the text is true.


## Challenge 3 · Verify fixture support

Implement `is_supported(claim, notes)`. A claim passes only if its source ID exists in the supplied notes and its text exactly matches that note. The claim is a validated Pydantic object. Treat paraphrases as unsupported in this deliberately narrow exercise.


In [ ]:
def is_supported(claim: BaseModel, notes: list[dict]) -> bool:
    return any(n["id"] == claim.source_id and n["text"] == claim.text for n in notes)


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
assert is_supported(parse_claim({"text": BY_ID["s3"]["text"], "source_id": "s3"}), NOTES)
assert not is_supported(parse_claim({"text": "Cedar costs one dollar.", "source_id": "s3"}), NOTES), "A real citation does not justify a false claim"
assert not is_supported(parse_claim({"text": BY_ID["s3"]["text"], "source_id": "missing"}), NOTES)
assert not is_supported(parse_claim({"text": "Mira is Cedar's maintainer.", "source_id": "s3"}), NOTES), "This checker intentionally rejects paraphrases"
print("PASS: support is checked separately from formatting")


### Why this works

The fixture check catches fabricated citations and unrelated claims with real citations. It does not validate the source itself or allow semantic paraphrases; those require additional evaluation.


## Optional · Connect a real model

This section is disabled by default and is not required for completion. To opt in, set `RUN_LIVE = True`, choose `OPENAI_MODEL`, and supply `OPENAI_API_KEY` through your environment. Choose a model supporting structured output. One invocation may incur a charge; retries are disabled. Never paste or print the key in a notebook.
The live schema is separate because provider schema support may differ from Pydantic's local constraints. Live-provider behavior was not tested for this workbook.


In [ ]:
RUN_LIVE = False
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "")

if RUN_LIVE:
    if not OPENAI_MODEL or not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_MODEL and OPENAI_API_KEY outside this notebook before opting in")
    from langchain_openai import ChatOpenAI

    class LiveClaim(BaseModel):
        text: str
        source_id: str

    model = ChatOpenAI(model=OPENAI_MODEL, timeout=30, max_retries=0)
    live_result = model.with_structured_output(LiveClaim).invoke(
        prepare_messages("Who maintains Cedar?", BY_ID["s3"])
    )
    checked = parse_claim(live_result.model_dump())
    print("Exact fixture support:", is_supported(checked, NOTES))
else:
    print("Skipped: optional paid model call is disabled")


## Reflection

1. Can an answer pass the schema and fail the evidence check? Give an example.
2. Why might a correct paraphrase fail this checker?
3. What additional check would detect a true but irrelevant statement?


### Discussion

1. A budget claim with source ID `s3` has the right shape but is not in that note.
2. Exact equality checks identity, not semantic equivalence.
3. Evaluate whether the supported claim answers the actual question. Source support and relevance are separate dimensions.


## References

- [Messages](https://docs.langchain.com/oss/python/langchain/messages)
- [Models](https://docs.langchain.com/oss/python/langchain/models)
- [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output)
